In [23]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('training_data_hieu_khai_niem.csv')

In [4]:
df.head()

,subject,concept,ground_truth,student_answer,label
question_id,,,,,
1,Vật lý 10,Định luật 1 Newton (quán tính),Nếu một vật không chịu tác dụng của lực nào ho...,Vật đang đứng yên hay đang chạy đều thì cứ giữ...,dung
2,Vật lý 10,Định luật 1 Newton (quán tính),Nếu một vật không chịu tác dụng của lực nào ho...,Em nghĩ là khi hợp lực tác dụng lên vật bằng 0...,dung
3,Vật lý 10,Định luật 1 Newton (quán tính),Nếu một vật không chịu tác dụng của lực nào ho...,Nếu một vật không chịu tác dụng của lực nào ho...,hoc_vet
4,Vật lý 10,Định luật 1 Newton (quán tính),Nếu một vật không chịu tác dụng của lực nào ho...,Một vật không chịu lực nào tác dụng hoặc hợp l...,hoc_vet
5,Vật lý 10,Định luật 1 Newton (quán tính),Nếu một vật không chịu tác dụng của lực nào ho...,Định luật 1 Newton nói là vật nào cũng cần có ...,sai


In [5]:
features = ['ground_truth', 'student_answer', 'label']
df = df[features]

In [6]:
from underthesea import word_tokenize
def tokenize(st: str) -> list:
    token = word_tokenize(st)
    token = [x.replace(' ', '_') for x in token]
    return token


In [7]:

df['tokenize_ground_truth'] = df['ground_truth'].apply(lambda st: tokenize(st))

In [8]:
df['tokenize_student_answer'] = df['student_answer'].apply(lambda st: tokenize(st))

In [9]:
df.head()

,ground_truth,student_answer,label,tokenize_ground_truth,tokenize_student_answer
question_id,,,,,
1,Nếu một vật không chịu tác dụng của lực nào ho...,Vật đang đứng yên hay đang chạy đều thì cứ giữ...,dung,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Vật, đang, đứng, yên, hay, đang, chạy, đều, t..."
2,Nếu một vật không chịu tác dụng của lực nào ho...,Em nghĩ là khi hợp lực tác dụng lên vật bằng 0...,dung,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Em, nghĩ, là, khi, hợp_lực, tác_dụng, lên, vậ..."
3,Nếu một vật không chịu tác dụng của lực nào ho...,Nếu một vật không chịu tác dụng của lực nào ho...,hoc_vet,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Nếu, một, vật, không, chịu, tác_dụng, của, lự..."
4,Nếu một vật không chịu tác dụng của lực nào ho...,Một vật không chịu lực nào tác dụng hoặc hợp l...,hoc_vet,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Một, vật, không, chịu, lực, nào, tác_dụng, ho..."
5,Nếu một vật không chịu tác dụng của lực nào ho...,Định luật 1 Newton nói là vật nào cũng cần có ...,sai,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Định, luật, 1, Newton, nói, là, vật, nào, cũn..."


In [10]:
from collections import Counter

def ngram_overlap(tokenize_ground_truth: list, tokenize_student_answer: list) -> float:
    bigram_gt = list(zip(tokenize_ground_truth, tokenize_ground_truth[1:]))
    trigram_gt = list(zip(tokenize_ground_truth, tokenize_ground_truth[1:], tokenize_ground_truth[2:]))
    
    bigram_sa = list(zip(tokenize_student_answer, tokenize_student_answer[1:]))
    trigram_sa = list(zip(tokenize_student_answer, tokenize_student_answer[1:], tokenize_student_answer[2:]))
    if not bigram_gt and not bigram_sa:
        return 0.0
    bigram_gtc = Counter(bigram_gt)
    trigram_gtc = Counter(trigram_gt)

    bigram_sac = Counter(bigram_sa)
    trigram_sac = Counter(trigram_sa)

    bigram_overlap = 0
    trigram_overlap = 0
    bigram_union = sum((bigram_gtc | bigram_sac).values())

    if bigram_union > 0:
        bigram_overlap = sum((bigram_gtc & bigram_sac).values()) / bigram_union

    trigram_union = sum((trigram_gtc | trigram_sac).values())
    if trigram_union > 0:
        trigram_overlap = sum((trigram_gtc & trigram_sac).values()) / trigram_union

    return (bigram_overlap + trigram_overlap) / 2


In [13]:
df['ngram_overlap'] = df.apply(lambda x: ngram_overlap(x['tokenize_ground_truth'], x['tokenize_student_answer']), axis=1)

In [15]:
df.head()

,ground_truth,student_answer,label,tokenize_ground_truth,tokenize_student_answer,ngram_overlap
question_id,,,,,,
1,Nếu một vật không chịu tác dụng của lực nào ho...,Vật đang đứng yên hay đang chạy đều thì cứ giữ...,dung,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Vật, đang, đứng, yên, hay, đang, chạy, đều, t...",0.028846
2,Nếu một vật không chịu tác dụng của lực nào ho...,Em nghĩ là khi hợp lực tác dụng lên vật bằng 0...,dung,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Em, nghĩ, là, khi, hợp_lực, tác_dụng, lên, vậ...",0.007353
3,Nếu một vật không chịu tác dụng của lực nào ho...,Nếu một vật không chịu tác dụng của lực nào ho...,hoc_vet,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Nếu, một, vật, không, chịu, tác_dụng, của, lự...",0.542398
4,Nếu một vật không chịu tác dụng của lực nào ho...,Một vật không chịu lực nào tác dụng hoặc hợp l...,hoc_vet,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Một, vật, không, chịu, lực, nào, tác_dụng, ho...",0.276423
5,Nếu một vật không chịu tác dụng của lực nào ho...,Định luật 1 Newton nói là vật nào cũng cần có ...,sai,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Định, luật, 1, Newton, nói, là, vật, nào, cũn...",0.000000


In [ ]:
def lcs(a: list, b: list) -> int:
    n, m = len(a), len(b)
    a, b = [0] + a, [0] + b
    pre = [0] * (m+5)
    cur = [0] * (m+5)
    for i in range(1, n+1):
        for j in range(1, m+1):
            if a[i] == b[j]:
                cur[j] = pre[j-1] + 1
            else:
                cur[j] = max(pre[j], cur[j-1])

        pre = cur[:]
    return cur[m]


In [ ]:
df['lcs_ratio'] = df.apply(lambda x: (lcs(x['tokenize_ground_truth'], x['tokenize_student_answer']) / len(x['tokenize_ground_truth'])), axis=1)

In [19]:
df.head()

,ground_truth,student_answer,label,tokenize_ground_truth,tokenize_student_answer,ngram_overlap,lcs_ratio
question_id,,,,,,,
1,Nếu một vật không chịu tác dụng của lực nào ho...,Vật đang đứng yên hay đang chạy đều thì cứ giữ...,dung,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Vật, đang, đứng, yên, hay, đang, chạy, đều, t...",0.028846,0.233333
2,Nếu một vật không chịu tác dụng của lực nào ho...,Em nghĩ là khi hợp lực tác dụng lên vật bằng 0...,dung,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Em, nghĩ, là, khi, hợp_lực, tác_dụng, lên, vậ...",0.007353,0.300000
3,Nếu một vật không chịu tác dụng của lực nào ho...,Nếu một vật không chịu tác dụng của lực nào ho...,hoc_vet,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Nếu, một, vật, không, chịu, tác_dụng, của, lự...",0.542398,0.866667
4,Nếu một vật không chịu tác dụng của lực nào ho...,Một vật không chịu lực nào tác dụng hoặc hợp l...,hoc_vet,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Một, vật, không, chịu, lực, nào, tác_dụng, ho...",0.276423,0.633333
5,Nếu một vật không chịu tác dụng của lực nào ho...,Định luật 1 Newton nói là vật nào cũng cần có ...,sai,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Định, luật, 1, Newton, nói, là, vật, nào, cũn...",0.000000,0.200000


In [20]:
df['length_ratio'] = df.apply(lambda x: len(x['tokenize_student_answer']) / len(x['tokenize_ground_truth']), axis=1)

In [21]:
df.head()

,ground_truth,student_answer,label,tokenize_ground_truth,tokenize_student_answer,ngram_overlap,lcs_ratio,length_ratio
question_id,,,,,,,,
1,Nếu một vật không chịu tác dụng của lực nào ho...,Vật đang đứng yên hay đang chạy đều thì cứ giữ...,dung,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Vật, đang, đứng, yên, hay, đang, chạy, đều, t...",0.028846,0.233333,0.900000
2,Nếu một vật không chịu tác dụng của lực nào ho...,Em nghĩ là khi hợp lực tác dụng lên vật bằng 0...,dung,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Em, nghĩ, là, khi, hợp_lực, tác_dụng, lên, vậ...",0.007353,0.300000,1.366667
3,Nếu một vật không chịu tác dụng của lực nào ho...,Nếu một vật không chịu tác dụng của lực nào ho...,hoc_vet,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Nếu, một, vật, không, chịu, tác_dụng, của, lự...",0.542398,0.866667,1.000000
4,Nếu một vật không chịu tác dụng của lực nào ho...,Một vật không chịu lực nào tác dụng hoặc hợp l...,hoc_vet,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Một, vật, không, chịu, lực, nào, tác_dụng, ho...",0.276423,0.633333,0.800000
5,Nếu một vật không chịu tác dụng của lực nào ho...,Định luật 1 Newton nói là vật nào cũng cần có ...,sai,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Định, luật, 1, Newton, nói, là, vật, nào, cũn...",0.000000,0.200000,0.900000


In [ ]:
def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    dot = np.dot(a, b)
    A = np.linalg.norm(a)
    B = np.linalg.norm(b)
    if A == 0 or B == 0:
        return .0
    return dot / (A * B)